In [1]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import spectral
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, cohen_kappa_score, classification_report
from tqdm.notebook import tqdm
from typing import Dict
from scipy.io import loadmat

In [2]:
DATASETS: Dict[str, Dict[str, object]] = {
    "PC": {
        "folder": "PC",
        "description": "Pavia Centre",
        "data_file": "Pavia.mat",
        "gt_file": "Pavia_gt.mat",
        "data_key": "pavia",
        "gt_key": "pavia_gt",
        "class_name": ['Water','Trees','Asphalt','Self-Blocking Bricks','Bitumen','Tiles','Shadows','Meadows','Bare Soil']
    },
    "PU": {
        "folder": "PU",
        "description": "Pavia University",
        "data_file": "PaviaU.mat",
        "gt_file": "PaviaU_gt.mat",
        "data_key": "paviaU",
        "gt_key": "paviaU_gt",
        "class_name": ['Asphalt','Meadows','Gravel','Trees','Painted metal sheets','Bare Soil','Bitumen','Self-Blocking Bricks','Shadows']
    },
}

In [3]:
COMMON_CLASSES_NAME = ["Asphalt", "Meadows", "Trees", "Bare Soil", "Bitumen", "Self-Blocking Bricks", "Shadows"]

COMMON_CLASSES = {
    1: "Asphalt",
    2: "Meadows",
    3: "Trees",
    4: "Bare Soil",
    5: "Bitumen",
    6: "Self-Blocking Bricks",
    7: "Shadows",
}

PC_TO_COMMON = {
    3: 1,  # Asphalt
    8: 2,  # Meadows
    2: 3,  # Trees
    9: 4,  # Bare Soil
    5: 5,  # Bitumen
    4: 6,  # Self-Blocking Bricks
    7: 7,  # Shadows
}

PU_TO_COMMON = {
    1: 1,  # Asphalt
    2: 2,  # Meadows
    4: 3,  # Trees
    6: 4,  # Bare Soil
    7: 5,  # Bitumen
    8: 6,  # Self-Blocking Bricks
    9: 7,  # Shadows
}

In [45]:
# Set hypeperameters and experimental settings just like the original code
RANDOM_SEED = 42
DATASET = 'PU'      # IP, PU, SA  
TRAIN_SIZE = 0.3    # ratio of training data
VAL_SIZE = 0.1      # ratio of valuating data
EPOCH = 30         # number of epoch
VAL_EPOCH = 5       # interval of valuation
LR = 0.001          # learning rate
WEIGHT_DECAY = 1e-4  
BATCH_SIZE = 256
DEVICE = 0          # -1:CPU  0:cuda 0
N_PCA = 15
NUM_CLASS = len(COMMON_CLASSES_NAME)
save_dir = Path("results")
save_dir.mkdir(parents=True, exist_ok=True)

# Set random seed just like the original code
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
DATA_DIR = Path("../data")

def load_hsi_data(dataset: str):
    entry = DATASETS[dataset]

    data_path = DATA_DIR / entry["folder"] / entry["data_file"]
    gt_path = DATA_DIR / entry["folder"] / entry["gt_file"]

    print("Loading:", data_path)
    print("Loading:", gt_path)

    data_mat = loadmat(data_path)
    gt_mat = loadmat(gt_path)

    X = data_mat[entry["data_key"]]
    y = gt_mat[entry["gt_key"]]

    return X, y

In [6]:
PC_X, PC_y = load_hsi_data('PC')
PU_X, PU_y = load_hsi_data('PU')

print(PC_X.shape, PC_y.shape)  # e.g. (145, 145, 200)
print(PU_X.shape, PU_y.shape)  # e.g. (145, 145)

Loading: ../data/PC/Pavia.mat
Loading: ../data/PC/Pavia_gt.mat
Loading: ../data/PU/PaviaU.mat
Loading: ../data/PU/PaviaU_gt.mat
(1096, 715, 102) (1096, 715)
(610, 340, 103) (610, 340)


In [7]:
# # Display HSI
# # RGB
# rgb_view = spectral.imshow(PC_X, (30, 20, 10), classes=PC_y, title="RGB origin", figsize=(7, 7))
# rgb_view.set_display_mode("data")
# #rgb_view.axes.figure.savefig(save_dir / f"{DATASET}_RGB_origin.jpg", bbox_inches="tight", dpi=300)

# # Ground truth
# gt_view = spectral.imshow(classes=PC_y, title="GroundTruth", figsize=(7, 7))
# #gt_view.axes.figure.savefig(save_dir / f"{DATASET}_gt.jpg", bbox_inches="tight", dpi=300)

# # Overlay
# view = spectral.imshow(PC_X, (30, 20, 10), classes=PC_y, figsize=(7, 7))
# view.set_display_mode("overlay")
# view.class_alpha = 0.5
# view.set_title("Overlay")
# #view.axes.figure.savefig(save_dir / f"{DATASET}_Overlay.jpg", bbox_inches="tight", dpi=300)

In [8]:
def remap_ground_truth(gt, mapping):
    remapped_gt = np.zeros_like(gt, dtype=np.int64)

    for original_label, common_label in mapping.items():
        remapped_gt[gt == original_label] = common_label

    return remapped_gt

# This remapped the classes and remove classes that only exist on one of the dataset
PC_y = remap_ground_truth(PC_y,PC_TO_COMMON)
PU_y = remap_ground_truth(PU_y,PU_TO_COMMON)

In [9]:
def compare_sample_counts(
    pc_gt: np.ndarray,
    pu_gt: np.ndarray,
    class_names: dict[int, str],
    pc_to_common: dict[int, str],
    pu_to_common: dict[int, str],
) -> pd.DataFrame:
    rows = []
    for class_id, class_name in class_names.items():
        pc_count = int(np.count_nonzero(pc_gt == class_id))
        pu_count = int(np.count_nonzero(pu_gt == class_id))

        pc_to_pu_percentage = (
            pu_count / pc_count * 100
            if pc_count > 0
            else np.nan
        )

        rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "PC_samples": pc_count,
            "PU_samples": pu_count,
            "PU_as_%_of_PC": pc_to_pu_percentage,
        })

    table = pd.DataFrame(rows)

    total_pc = int(table["PC_samples"].sum())
    total_pu = int(table["PU_samples"].sum())

    overall_percentage = (
        total_pu / total_pc * 100
        if total_pu > 0
        else np.nan
    )

    total_row = pd.DataFrame([{
        "class_id": "Total",
        "class_name": "All shared classes",
        "PC_samples": total_pc,
        "PU_samples": total_pu,
        "PU_as_%_of_PC": overall_percentage,
    }])

    return pd.concat(
        [table, total_row],
        ignore_index=True,
    )

In [10]:
sample_comparison = compare_sample_counts(
    pc_gt=PC_y,
    pu_gt=PU_y,
    class_names=COMMON_CLASSES,
    pc_to_common=PC_TO_COMMON,
    pu_to_common=PU_TO_COMMON,
)

print(
    sample_comparison.to_string(
        index=False,
        formatters={
            "PU_as_%_of_PC": lambda value: f"{value:.2f}%"
        },
    )
)

class_id           class_name  PC_samples  PU_samples PU_as_%_of_PC
       1              Asphalt        3090        6631       214.60%
       2              Meadows       42826       18649        43.55%
       3                Trees        7598        3064        40.33%
       4            Bare Soil        2863        5029       175.65%
       5              Bitumen        6584        1330        20.20%
       6 Self-Blocking Bricks        2685        3682       137.13%
       7              Shadows        7287         947        13.00%
   Total   All shared classes       72933       39332        53.93%


In [11]:
def extract_labelled_pixels(X, y, background_label=0):
    """
    X: hyperspectral image with shape (height, width, bands)
    y: label map with shape (height, width)

    Returns:
        X_pixels: (number_of_labelled_pixels, bands)
        y_pixels: (number_of_labelled_pixels,)
    """
    if X.ndim != 3:
        raise ValueError(
            f"X must have shape (height, width, bands), got {X.shape}"
        )

    if y.ndim != 2:
        raise ValueError(
            f"y must have shape (height, width), got {y.shape}"
        )

    if X.shape[:2] != y.shape:
        raise ValueError(
            f"Spatial dimensions do not match: "
            f"X={X.shape[:2]}, y={y.shape}"
        )

    bands = X.shape[-1]

    X_pixels = X.reshape(-1, bands)
    y_pixels = y.reshape(-1)

    # Keep only labelled pixels
    labelled_mask = y_pixels != background_label

    X_pixels = X_pixels[labelled_mask]
    y_pixels = y_pixels[labelled_mask]

    return X_pixels.astype(np.float32), y_pixels.astype(np.int64)

In [12]:
PC_X_pixels, PC_y_pixels = extract_labelled_pixels(PC_X, PC_y)
PU_X_pixels, PU_y_pixels = extract_labelled_pixels(PU_X, PU_y)

print("PC:", PC_X_pixels.shape, PC_y_pixels.shape)
print("PU:", PU_X_pixels.shape, PU_y_pixels.shape)

PC: (72933, 102) (72933,)
PU: (39332, 103) (39332,)


In [13]:
# Convert labels from 1...C to 0...C-1
PC_y_pixels = PC_y_pixels - 1
PU_y_pixels = PU_y_pixels - 1

X_train, y_train = PU_X_pixels, PU_y_pixels

X_val, X_test, y_val, y_test = train_test_split(
    PC_X_pixels, PC_y_pixels,
    train_size=(VAL_SIZE * (len(PC_y_pixels) + len(PU_y_pixels)))/len(PC_y_pixels),
    random_state=RANDOM_SEED,
    stratify=PC_y_pixels
)

In [14]:
def apply_pca(X_train, n_components):
    """
    Apply PCA.

    Parameters
    ----------
    X_train : np.ndarray
        Shape: (number_of_training_pixels, number_of_bands)
    n_components : int
        Number of PCA components to retain.

    Returns
    -------
    X_train_pca : np.ndarray
        PCA-transformed training data.
    scaler : StandardScaler
        Fitted training-data scaler.
    pca : PCA
        Fitted PCA object.
    """
    pca = PCA(
        n_components=n_components,
        random_state=RANDOM_SEED
    )

    # Fit PCA only on training data
    X_train_pca = pca.fit_transform(X_train)

    return X_train_pca.astype(np.float32), pca

In [15]:
# Fit PCA only on the training data
# X_train, pca = apply_pca(X_train, n_components=N_PCA)
# X_val = pca.fit_transform(X_val)
# X_test = pca.fit_transform(X_test)
print(X_train.shape)  # (39332, 15)

(39332, 103)


In [16]:
y_all = np.concatenate([
    y_train,
    y_val,
    y_test
])

# --------------------------------------------------
# Overall split information
# --------------------------------------------------
def print_dataset_split_summary(y_train, y_val, y_test):
    train_count = len(y_train)
    val_count = len(y_val)
    test_count = len(y_test)

    total_count = train_count + val_count + test_count

    print("\n" + "=" * 65)
    print("DATASET SPLIT SUMMARY")
    print("=" * 65)

    print(
        f"{'Dataset':15s} | {'Samples':>10s} | "
        f"{'Percentage of all data':>22s}"
    )
    print("-" * 65)

    print(
        f"{'Train':15s} | {train_count:10d} | "
        f"{train_count / total_count * 100:21.2f}%"
    )
    print(
        f"{'Validation':15s} | {val_count:10d} | "
        f"{val_count / total_count * 100:21.2f}%"
    )
    print(
        f"{'Test':15s} | {test_count:10d} | "
        f"{test_count / total_count * 100:21.2f}%"
    )
    print("-" * 65)
    print(f"{'All data':15s} | {total_count:10d} | {100:21.2f}%")

    # Also show how PU alone was divided
    pu_total = val_count + test_count

    print("\nPU VALIDATION/TEST SPLIT")
    print("-" * 65)
    print(
        f"Validation: {val_count:6d} samples "
        f"({val_count / pu_total * 100:.2f}% of PU)"
    )
    print(
        f"Test:       {test_count:6d} samples "
        f"({test_count / pu_total * 100:.2f}% of PU)"
    )


# --------------------------------------------------
# Per-class distribution
# --------------------------------------------------
import numpy as np


def print_class_distribution(
    labels,
    all_labels,
    class_names,
    title
):
    labels = np.asarray(labels).reshape(-1)
    all_labels = np.asarray(all_labels).reshape(-1)

    num_classes = len(class_names)

    # Class counts in the current split
    split_counts = np.bincount(
        labels,
        minlength=num_classes
    )

    # Total count of each class across train + validation + test
    total_class_counts = np.bincount(
        all_labels,
        minlength=num_classes
    )

    split_total = len(labels)

    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)

    print(
        f"{'ID':>3s} | "
        f"{'Class':30s} | "
        f"{'Split samples':>13s} | "
        f"{'Total class':>11s} | "
        f"{'% of class':>10s}"
    )
    print("-" * 95)

    for class_id in range(num_classes):
        split_count = split_counts[class_id]
        total_class_count = total_class_counts[class_id]

        classwise_percentage = (
            split_count / total_class_count * 100
            if total_class_count > 0
            else 0.0
        )

        print(
            f"{class_id:3d} | "
            f"{class_names[class_id]:30s} | "
            f"{split_count:13d} | "
            f"{total_class_count:11d} | "
            f"{classwise_percentage:9.2f}%"
        )

    print("-" * 75)
    print(
        f"{'Total split samples':>49s} | "
        f"{split_total:13d}"
    )

# --------------------------------------------------
# Print results
# --------------------------------------------------
print_dataset_split_summary(
    y_train,
    y_val,
    y_test
)

total_size = len(y_train) + len(y_val) + len(y_test)

print_class_distribution(
    y_train,
    y_all,
    COMMON_CLASSES_NAME,
    "TRAIN SET — PC DATA"
)

print_class_distribution(
    y_val,
    y_all,
    COMMON_CLASSES_NAME,
    "VALIDATION SET — PU DATA"
)

print_class_distribution(
    y_test,
    y_all,
    COMMON_CLASSES_NAME,
    "TEST SET — PU DATA"
)


DATASET SPLIT SUMMARY
Dataset         |    Samples | Percentage of all data
-----------------------------------------------------------------
Train           |      39332 |                 35.03%
Validation      |      11226 |                 10.00%
Test            |      61707 |                 54.97%
-----------------------------------------------------------------
All data        |     112265 |                100.00%

PU VALIDATION/TEST SPLIT
-----------------------------------------------------------------
Validation:  11226 samples (15.39% of PU)
Test:        61707 samples (84.61% of PU)

TRAIN SET — PC DATA
 ID | Class                          | Split samples | Total class | % of class
-----------------------------------------------------------------------------------------------
  0 | Asphalt                        |          6631 |        9721 |     68.21%
  1 | Meadows                        |         18649 |       61475 |     30.34%
  2 | Trees                          |    

In [17]:
class HSIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        spectrum = self.X[index]

        # Convert (bands,) into (1, bands)
        spectrum = spectrum.unsqueeze(0)

        label = self.y[index]

        return spectrum, label

In [41]:
train_dataset = HSIDataset(X_train, y_train)
val_dataset = HSIDataset(X_val, y_val)
test_dataset = HSIDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [42]:
spectrum, label = train_dataset[0]

print("Spectrum shape:", spectrum.shape)
print("Label shape:", label.shape)
print("Label value:", label.item())
print("Spectrum values:")
print(spectrum)

Spectrum shape: torch.Size([1, 103])
Label shape: torch.Size([])
Label value: 0
Spectrum values:
tensor([[1447., 1113.,  973., 1053., 1180., 1263., 1285., 1320., 1283., 1295.,
         1337., 1317., 1327., 1371., 1404., 1430., 1461., 1470., 1494., 1519.,
         1523., 1542., 1590., 1608., 1587., 1606., 1631., 1637., 1658., 1697.,
         1727., 1723., 1743., 1787., 1793., 1817., 1841., 1862., 1881., 1894.,
         1907., 1905., 1892., 1906., 1915., 1915., 1933., 1935., 1928., 1913.,
         1925., 1949., 1958., 1954., 1937., 1922., 1939., 1949., 1945., 1951.,
         1957., 1963., 1973., 1955., 1914., 1925., 1956., 1953., 1978., 1978.,
         1944., 1922., 1918., 1937., 1946., 1941., 1940., 1936., 1900., 1904.,
         1917., 1899., 1845., 1827., 1871., 1869., 1851., 1827., 1821., 1820.,
         1830., 1829., 1807., 1793., 1788., 1749., 1696., 1708., 1714., 1698.,
         1660., 1643., 1610.]])


1D CNN in theory should work better without PCA, because then the model would learn the spectral relationship of the data.

- 1D CNN on original bands: learns local spectral patterns.
- 1D CNN on PCA components: learns local patterns across ordered components.
- MLP on PCA components: often makes more conceptual sense because it does not assume local adjacency.

In [43]:
class CNN1D(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.4):
        super().__init__()

        self.conv1d_1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )

        self.conv1d_2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )

        self.conv1d_3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(128, num_classes),
        )

    def _forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1d_1(x)  # (Batch, 32, 7)
        x = self.conv1d_2(x)  # (Batch, 64, 3)
        x = self.conv1d_3(x)  # (Batch, 128, 1)

        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self._forward_features(x)
        x = self.classifier(x)
        return x

In [21]:
model = CNN1D(NUM_CLASS)

summary(
    model,
    input_size=(BATCH_SIZE, 1, N_PCA),  # batch, channel, bands/components
    col_names=("num_params", "kernel_size", "mult_adds", "input_size", "output_size"),
    col_width=14,
    row_settings=("var_names",),
    depth=4,
)

Layer (type (var_name))                  Param #        Kernel Shape   Mult-Adds      Input Shape    Output Shape
CNN1D (CNN1D)                            --             --             --             [256, 1, 15]   [256, 7]
├─Sequential (conv1d_1)                  --             --             --             [256, 1, 15]   [256, 32, 7]
│    └─Conv1d (0)                        192            [5]            737,280        [256, 1, 15]   [256, 32, 15]
│    └─ReLU (1)                          --             --             --             [256, 32, 15]  [256, 32, 15]
│    └─MaxPool1d (2)                     --             2              --             [256, 32, 15]  [256, 32, 7]
├─Sequential (conv1d_2)                  --             --             --             [256, 32, 7]   [256, 64, 3]
│    └─Conv1d (0)                        6,208          [3]            11,124,736     [256, 32, 7]   [256, 64, 7]
│    └─ReLU (1)                          --             --             --             [256

In [22]:
X_batch, y_batch = next(iter(train_loader))

print("Original batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)

X_batch = X_batch.to(DEVICE)
y_batch = y_batch.to(DEVICE)

with torch.no_grad():
    output = model(X_batch)

print("Model output shape:", output.shape)

Original batch shape: torch.Size([256, 1, 103])
Label batch shape: torch.Size([256])
Model output shape: torch.Size([256, 7])


In [23]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Remove gradients left from the previous batch
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch)

        # Calculate loss
        loss = criterion(logits, y_batch)

        # Calculate gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [24]:
def evaluate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            running_loss += loss.item() * X_batch.size(0)

            predictions = logits.argmax(dim=1)

            correct += (
                predictions == y_batch
            ).sum().item()

            total += y_batch.size(0)

    average_loss = running_loss / total
    accuracy = correct / total

    return average_loss, accuracy

In [46]:
device = torch.device(f"cuda:{DEVICE}" if DEVICE >= 0 and torch.cuda.is_available() else "cpu")

model = CNN1D(
    NUM_CLASS,
    dropout=0.5,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_accuracy = 0.0
best_model_path = save_dir / "best_model.pth"

for epoch in tqdm(range(1, EPOCH + 1)):
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_accuracy = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device
    )

    print(
        f"Epoch {epoch:03d}/{EPOCH} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print(
            f"Saved new best model: "
            f"{best_val_accuracy:.4f}"
        )
print(
    f"Saved new best model: "
    f"{best_val_accuracy:.4f}"
)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 001/30 | Train Loss: 8.0271 | Train Acc: 0.5912 | Val Loss: 14.7180 | Val Acc: 0.5872
Saved new best model: 0.5872
Epoch 002/30 | Train Loss: 1.4871 | Train Acc: 0.5762 | Val Loss: 63.9734 | Val Acc: 0.5872
Epoch 003/30 | Train Loss: 2.9478 | Train Acc: 0.4693 | Val Loss: 11.8294 | Val Acc: 0.5872
Epoch 004/30 | Train Loss: 1.5968 | Train Acc: 0.4801 | Val Loss: 7.4796 | Val Acc: 0.5872
Epoch 005/30 | Train Loss: 1.4183 | Train Acc: 0.4709 | Val Loss: 29.0459 | Val Acc: 0.5872
Epoch 006/30 | Train Loss: 1.8820 | Train Acc: 0.4848 | Val Loss: 78.2671 | Val Acc: 0.5872
Epoch 007/30 | Train Loss: 2.8067 | Train Acc: 0.4245 | Val Loss: 2.2136 | Val Acc: 0.5872
Epoch 008/30 | Train Loss: 1.7377 | Train Acc: 0.4749 | Val Loss: 1.7599 | Val Acc: 0.5881
Saved new best model: 0.5881
Epoch 009/30 | Train Loss: 1.7076 | Train Acc: 0.4727 | Val Loss: 1.7186 | Val Acc: 0.5878
Epoch 010/30 | Train Loss: 1.6699 | Train Acc: 0.4731 | Val Loss: 1.6908 | Val Acc: 0.5879
Epoch 011/30 | Train Loss: 

In [37]:
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model.to(device)
model.eval()

CNN1D(
  (conv1d_1): Sequential(
    (0): Conv1d(1, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv1d_2): Sequential(
    (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv1d_3): Sequential(
    (0): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=128, out_features=7, bias=True)
  )
)

In [38]:
test_loss, test_accuracy = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Test loss: 1.8333
Test accuracy: 0.6761


In [39]:
all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        predictions = logits.argmax(dim=1)

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            y_batch.cpu()
        )

y_pred = torch.cat(all_predictions).numpy()
y_true = torch.cat(all_targets).numpy()

print("Predictions shape:", y_pred.shape)
print("Targets shape:", y_true.shape)

Predictions shape: (61707,)
Targets shape: (61707,)


In [40]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    cohen_kappa_score,
    classification_report
)

oa = accuracy_score(y_true, y_pred)

aa = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred
)

print(f"Overall Accuracy: {oa:.4f}")
print(f"Average Accuracy: {aa:.4f}")
print(f"Kappa: {kappa:.4f}")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=COMMON_CLASSES_NAME,
        zero_division=0
    )
)

Overall Accuracy: 0.6761
Average Accuracy: 0.3191
Kappa: 0.4339
                      precision    recall  f1-score   support

             Asphalt       0.00      0.00      0.00      2614
             Meadows       0.83      0.98      0.90     36234
               Trees       0.80      0.78      0.79      6429
           Bare Soil       0.00      0.00      0.00      2422
             Bitumen       0.00      0.00      0.00      5571
Self-Blocking Bricks       0.40      0.47      0.43      2272
             Shadows       0.00      0.00      0.00      6165

            accuracy                           0.68     61707
           macro avg       0.29      0.32      0.30     61707
        weighted avg       0.59      0.68      0.63     61707



In [50]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def extract_cnn_features(model, loader, device):
    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)

            features = model._forward_features(X_batch)

            # Convert shape from (batch, 128, 1) to (batch, 128)
            features = features.view(features.size(0), -1)

            all_features.append(features.cpu().numpy())
            all_labels.append(y_batch.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    return all_features, all_labels

In [48]:
X_train_features, y_train_svm = extract_cnn_features(
    model,
    train_loader,
    device
)

X_val_features, y_val_svm = extract_cnn_features(
    model,
    val_loader,
    device
)

X_test_features, y_test_svm = extract_cnn_features(
    model,
    test_loader,
    device
)

In [51]:
svm_classifier = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced"
    )
)

svm_classifier.fit(X_train_features, y_train_svm)

,steps,"[('standardscaler', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,10
,kernel,'rbf'
,degree,3
,gamma,'scale'


In [52]:
y_pred_svm = svm_classifier.predict(X_test_features)

oa = accuracy_score(y_test_svm, y_pred_svm)

aa = recall_score(
    y_test_svm,
    y_pred_svm,
    average="macro",
    zero_division=0
)

kappa = cohen_kappa_score(
    y_test_svm,
    y_pred_svm
)

print(f"SVM Overall Accuracy: {oa:.4f}")
print(f"SVM Average Accuracy: {aa:.4f}")
print(f"SVM Kappa: {kappa:.4f}")

print(
    classification_report(
        y_test_svm,
        y_pred_svm,
        target_names=COMMON_CLASSES_NAME,
        zero_division=0
    )
)

SVM Overall Accuracy: 0.3192
SVM Average Accuracy: 0.1118
SVM Kappa: -0.0823
                      precision    recall  f1-score   support

             Asphalt       0.00      0.00      0.00      2614
             Meadows       0.52      0.49      0.51     36234
               Trees       0.02      0.07      0.03      6429
           Bare Soil       0.05      0.00      0.00      2422
             Bitumen       0.00      0.00      0.00      5571
Self-Blocking Bricks       0.00      0.00      0.00      2272
             Shadows       0.22      0.22      0.22      6165

            accuracy                           0.32     61707
           macro avg       0.12      0.11      0.11     61707
        weighted avg       0.33      0.32      0.32     61707

